In [1]:
from typing import Optional, List

class AvlNode:
    __slots__ = ("value", "l_child", "r_child", "depth")
    def __init__(self, num: int):
        self.value = num
        self.l_child: Optional[AvlNode] = None
        self.r_child: Optional[AvlNode] = None
        self.depth = 1

class AvlBalanceTree:
    def __init__(self):
        self.top: Optional[AvlNode] = None
        self.rotate_times = 0

    def get_node_depth(self, node: Optional[AvlNode]) -> int:
        return node.depth if node else 0

    def calc_balance_factor(self, node: Optional[AvlNode]) -> int:
        if not node:
            return 0
        return self.get_node_depth(node.l_child) - self.get_node_depth(node.r_child)

    def refresh_depth(self, node: Optional[AvlNode]) -> None:
        if node:
            node.depth = 1 + max(self.get_node_depth(node.l_child), self.get_node_depth(node.r_child))

    def turn_right(self, core: AvlNode) -> AvlNode:
        print(f"旋转操作：右旋LL，节点{core.value}下移")
        left_sub = core.l_child
        trans_sub = left_sub.r_child

        left_sub.r_child = core
        core.l_child = trans_sub

        self.refresh_depth(core)
        self.refresh_depth(left_sub)
        self.rotate_times += 1
        return left_sub

    def turn_left(self, core: AvlNode) -> AvlNode:
        print(f"旋转操作：左旋RR，节点{core.r_child.value}提升为上层")
        right_sub = core.r_child
        trans_sub = right_sub.l_child

        right_sub.l_child = core
        core.r_child = trans_sub

        self.refresh_depth(core)
        self.refresh_depth(right_sub)
        self.rotate_times += 1
        return right_sub

    def add_recur(self, node: Optional[AvlNode], num: int) -> AvlNode:
        if not node:
            return AvlNode(num)
        if num < node.value:
            node.l_child = self.add_recur(node.l_child, num)
        elif num > node.value:
            node.r_child = self.add_recur(node.r_child, num)
        else:
            return node

        self.refresh_depth(node)
        bf = self.calc_balance_factor(node)
        print(f"节点{node.value} | 当前深度:{node.depth} 平衡因子:{bf}")

        # LL失衡
        if bf > 1 and num < node.l_child.value:
            print(f"检测失衡：节点{node.value} LL结构，执行右旋")
            return self.turn_right(node)
        # RR失衡
        if bf < -1 and num > node.r_child.value:
            print(f"检测失衡：节点{node.value} RR结构，执行左旋")
            return self.turn_left(node)
        # LR失衡
        if bf > 1 and num > node.l_child.value:
            print(f"检测失衡：节点{node.value} LR结构，先左后右双旋")
            node.l_child = self.turn_left(node.l_child)
            return self.turn_right(node)
        # RL失衡
        if bf < -1 and num < node.r_child.value:
            print(f"检测失衡：节点{node.value} RL结构，先右后左双旋")
            node.r_child = self.turn_right(node.r_child)
            return self.turn_left(node)
        return node

    def add_number(self, num: int) -> None:
        self.top = self.add_recur(self.top, num)

    def show_detail_info(self) -> None:
        print("\n节点详情(数值[深度,平衡因子])：")
        self._traverse_detail(self.top, 0)

    def _traverse_detail(self, node: Optional[AvlNode], level: int) -> None:
        if node:
            blank = "  " * level
            bf = self.calc_balance_factor(node)
            print(f"{blank}{node.value}[d={node.depth}, bf={bf}]")
            self._traverse_detail(node.l_child, level + 1)
            self._traverse_detail(node.r_child, level + 1)

    def mid_order_collect(self, node: Optional[AvlNode], res: List[int] = None) -> List[int]:
        if res is None:
            res = []
        if node:
            self.mid_order_collect(node.l_child, res)
            res.append(node.value)
            self.mid_order_collect(node.r_child, res)
        return res

    def draw_tree_layout(self) -> None:
        if not self.top:
            print("当前无节点，树为空")
            return
        print("\n树形布局展示：")
        layout_lines = self.generate_layout(self.top)
        for line in layout_lines:
            print(line)

    def generate_layout(self, node: Optional[AvlNode]) -> List[str]:
        if not node:
            return []
        text = str(node.value)
        left_lines = self.generate_layout(node.l_child)
        right_lines = self.generate_layout(node.r_child)

        if not left_lines and not right_lines:
            block_w = max(4, len(text) + 2)
            space_prefix = " " * ((block_w - len(text)) // 2)
            return [space_prefix + text + " " * (block_w - len(space_prefix) - len(text))]
        if not right_lines:
            base_w = len(left_lines[0])
            pad = " " * ((base_w - len(text)) // 2)
            head = pad + text + " " * (base_w - len(pad) - len(text))
            return [head] + left_lines
        if not left_lines:
            base_w = len(right_lines[0])
            pad = " " * ((base_w - len(text)) // 2)
            head = pad + text + " " * (base_w - len(pad) - len(text))
            return [head] + right_lines

        left_width = len(left_lines[0])
        right_width = len(right_lines[0])
        gap = 3
        full_width = left_width + gap + right_width
        mid_pad = " " * ((full_width - len(text)) // 2)
        top_line = mid_pad + text + " " * (full_width - len(mid_pad) - len(text))

        merge_lines = []
        max_h = max(len(left_lines), len(right_lines))
        for idx in range(max_h):
            l_part = left_lines[idx] if idx < len(left_lines) else " " * left_width
            r_part = right_lines[idx] if idx < len(right_lines) else " " * right_width
            merge_lines.append(l_part + " " * gap + r_part)
        return [top_line] + merge_lines

    def global_balance_check(self, node: Optional[AvlNode]) -> tuple[bool, int]:
        if not node:
            return True, 0
        left_ok, left_d = self.global_balance_check(node.l_child)
        right_ok, right_d = self.global_balance_check(node.r_child)
        balance_flag = left_ok and right_ok and abs(left_d - right_d) <= 1
        return balance_flag, 1 + max(left_d, right_d)

if __name__ == "__main__":
    dividing_line = "-" * 60
    print(dividing_line)
    print("AVL平衡树分步构建演示")
    input_seq = [30, 20, 10, 25, 40, 35, 50]
    print(f"待插入数值序列：{input_seq}")
    print(dividing_line)

    tree = AvlBalanceTree()
    for step_idx, val in enumerate(input_seq):
        print(f"\n{dividing_line}")
        print(f"第{step_idx + 1}轮插入数值：{val}")
        print(dividing_line)
        tree.add_number(val)

        tree.show_detail_info()
        tree.draw_tree_layout()

        mid_res = tree.mid_order_collect(tree.top)
        print(f"\n中序遍历结果：{mid_res}")

        balance_state, tree_max_depth = tree.global_balance_check(tree.top)
        state_text = "平衡正常" if balance_state else "存在失衡节点"
        print(f"当前树最大深度：{tree_max_depth}，整体平衡状态：{state_text}")

    print(f"\n{dividing_line}")
    print("AVL树构建完成 全局汇总信息")
    print(dividing_line)
    print(f"全程旋转总次数：{tree.rotate_times}")
    final_mid = tree.mid_order_collect(tree.top)
    print(f"最终完整中序遍历：{final_mid}")

    # 校验二叉搜索树有序性
    is_valid_bst = True
    for i in range(len(final_mid) - 1):
        if final_mid[i] >= final_mid[i+1]:
            is_valid_bst = False
            break
    print(f"BST有序性校验：{'通过' if is_valid_bst else '不满足'}")

    final_bal, final_dep = tree.global_balance_check(tree.top)
    print(f"AVL平衡约束校验：{'通过' if final_bal else '不满足'}")
    print(f"最终树整体深度：{final_dep}")

------------------------------------------------------------
AVL平衡树分步构建演示
待插入数值序列：[30, 20, 10, 25, 40, 35, 50]
------------------------------------------------------------

------------------------------------------------------------
第1轮插入数值：30
------------------------------------------------------------

节点详情(数值[深度,平衡因子])：
30[d=1, bf=0]

树形布局展示：
 30 

中序遍历结果：[30]
当前树最大深度：1，整体平衡状态：平衡正常

------------------------------------------------------------
第2轮插入数值：20
------------------------------------------------------------
节点30 | 当前深度:2 平衡因子:1

节点详情(数值[深度,平衡因子])：
30[d=2, bf=1]
  20[d=1, bf=0]

树形布局展示：
 30 
 20 

中序遍历结果：[20, 30]
当前树最大深度：2，整体平衡状态：平衡正常

------------------------------------------------------------
第3轮插入数值：10
------------------------------------------------------------
节点20 | 当前深度:2 平衡因子:1
节点30 | 当前深度:3 平衡因子:2
检测失衡：节点30 LL结构，执行右旋
旋转操作：右旋LL，节点30下移

节点详情(数值[深度,平衡因子])：
20[d=2, bf=0]
  10[d=1, bf=0]
  30[d=1, bf=0]

树形布局展示：
    20     
 10     30 

中序遍历结果：[10, 20, 30]
当前树最大深度：2，整体平衡状